In [ ]:
# autorelead libraries
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch sees:", torch.cuda.device_count(), "GPUs")
print("Current device index:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name())


In [ ]:
import time
import random
import pickle
import argparse

import numpy as np
import scipy.sparse as sp
import scipy.signal as sig
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader

from config import get_config
from dataset import data_loader
from neural_methods.model.RadarNet import RadarNet

In [ ]:
def calcMetrics(input_data, sampFreq, B, window_size=None, use_harmonic=False, normalize=False):
    # define goodness parameters
    B1 = 0.75 # low cutoff
    B2 = 3 # high cutoff

    if window_size is None:
        pulseCalcLength = len(input_data)
    else:
        pulseCalcLength = np.uint32(window_size/(1.0/sampFreq))
    pulseRate_result = np.empty(len(input_data)-pulseCalcLength + 1)
    goodnessMetric_result = np.empty(len(input_data)-pulseCalcLength + 1)

    for i in range(0, len(pulseRate_result)):
        window = input_data[i:i+pulseCalcLength]
        # ppgFreq, ppgPower = sig.periodogram(window, fs=sampFreq, nfft=180000)
        ppgFreq, ppgPower = sig.welch(x=window, nperseg=len(window)//3, fs=sampFreq, nfft=180000)
        maskFreq = (ppgFreq > B1)&(ppgFreq<B2)
        # ppgFreq = ppgFreq[maskFreq]
        ppgPower = ppgPower * maskFreq
        if use_harmonic:
            # harmonic PSD
            harmonicppgPower = ppgPower[::2]
            harmonicppgPower = np.pad(harmonicppgPower, (0, len(ppgPower) - len(harmonicppgPower)))
            # find peak frequency
            peakFreq = ppgFreq[np.argmax(ppgPower + harmonicppgPower)]
        else:
            peakFreq = ppgFreq[np.argmax(ppgPower)]
        pulseRate = 60.0*peakFreq
        pulseRate_result[i] = pulseRate
        # compute goodness
        aroundPulseRate = (ppgFreq > peakFreq - B) & (ppgFreq < peakFreq + B)
        withinBandpass = (ppgFreq >= B1) & (ppgFreq <= B2)
        powerPulseRate = np.sum(ppgPower[aroundPulseRate])
        powerAll = np.sum(ppgPower[withinBandpass])
        if normalize:
            goodnessMetric_result[i] = powerPulseRate / powerAll
        else:
            goodnessMetric_result[i] = powerPulseRate / (powerAll - powerPulseRate)
    timestamps = np.arange(len(pulseRate_result))
    return timestamps, goodnessMetric_result, pulseRate_result

def custom_detrend(sig, Lambda):
    """custom_detrend(sig, Lambda) -> filtered_signal
    This function applies a detrending filter.
    This code is based on the following article "An advanced detrending method with application
    to HRV analysis". Tarvainen et al., IEEE Trans on Biomedical Engineering, 2002.
    *Parameters*
      ``sig`` (1d numpy array):
        The sig where you want to remove the trend.
      ``Lambda`` (int):
        The smoothing parameter.
    *Returns*
      ``filtered_signal`` (1d numpy array):
        The detrended sig.
    """
    signal_length = sig.shape[0]

    # observation matrix
    H = np.identity(signal_length)

    # second-order difference matrix

    ones = np.ones(signal_length)
    minus_twos = -2 * np.ones(signal_length)
    diags_data = np.array([ones, minus_twos, ones])
    diags_index = np.array([0, 1, 2])
    D = sp.spdiags(diags_data, diags_index, (signal_length - 2), signal_length).toarray()
    filtered_signal = np.dot((H - np.linalg.inv(H + (Lambda ** 2) * np.dot(D.T, D))), sig)
    return filtered_signal

def pulse_rate_from_power_spectral_density(pleth_sig: np.array, FS: float,
                                           LL_PR: float, UL_PR: float,
                                           BUTTER_ORDER: int = 6,
                                           DETREND: bool = False,
                                           FResBPM: float = 0.1,
                                           HARMONIC: bool = False,
                                           WELCH = True) -> float:
    """ Function to estimate the pulse rate from the power spectral density of the plethysmography sig.

    Args:
        pleth_sig (np.array): Plethysmography sig.
        FS (float): Sampling frequency.
        LL_PR (float): Lower cutoff frequency for the butterworth filtering.
        UL_PR (float): Upper cutoff frequency for the butterworth filtering.
        BUTTER_ORDER (int, optional): Order of the butterworth filter. Give None to skip filtering. Defaults to 6.
        DETREND (bool, optional): Boolena Flag for executing cutsom_detrend. Defaults to False.
        FResBPM (float, optional): Frequency resolution. Defaults to 0.1.

    Returns:
        pulse_rate (float): _description_
    

    Daniel McDuff, Ethan Blackford, January 2019
    Copyright (c)
    Licensed under the MIT License and the RAIL AI License.
    """

    N = (60*FS)/FResBPM

    # Detrending + nth order butterworth + periodogram
    if DETREND:
        pleth_sig = custom_detrend(pleth_sig, 100)
    if BUTTER_ORDER:
        [b, a] = sig.butter(BUTTER_ORDER, [LL_PR/60, UL_PR/60], btype='bandpass', fs = FS)
        pleth_sig = sig.filtfilt(b, a, np.double(pleth_sig))
    
    # Calculate the PSD and the mask for the desired range
    if WELCH:
        F, Pxx = sig.welch(x=pleth_sig, nperseg=len(pleth_sig)//3, nfft=N, fs=FS)
    else:
        F, Pxx = sig.periodogram(x=pleth_sig,  nfft=N, fs=FS);  
    FMask = (F >= (LL_PR/60)) & (F <= (UL_PR/60))
    
    # Calculate predicted pulse rate:
    FRange = F * FMask
    PRange = Pxx * FMask

    if HARMONIC:
      harmonicppgPower = PRange[::2]
      harmonicppgPower = np.pad(harmonicppgPower, (0, len(PRange) - len(harmonicppgPower)))
      harmonicppgPower[0] = 0
      MaxInd = np.argmax(PRange+harmonicppgPower)
    else:
      MaxInd = np.argmax(PRange)
    pulse_rate_freq = FRange[MaxInd]
    pulse_rate = pulse_rate_freq*60
            
    return pulse_rate

In [ ]:
class Args:
    config_file = 'configs/train_configs/CogPhys_Resp_Radar_BASIC.yaml'
    cached_path = None
    preprocess = None
    lr = None
    model_file_name = None

args = Args()
config = get_config(args)
# print('Configuration:')
# print(config, end='\n\n')
data_loader_dict = dict() # dictionary of data loaders 
train_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
train_data_loader = train_loader(
    name="train",
    data_path=config.TRAIN.DATA.DATA_PATH,
    config_data=config.TRAIN.DATA,
    device=config.DEVICE)
data_loader_dict['train'] = DataLoader(
    dataset=train_data_loader,
    num_workers=2,
    batch_size=2,
    shuffle=True,
)
print(); print()

valid_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
valid_data_loader = valid_loader(
    name="valid",
    data_path=config.VALID.DATA.DATA_PATH,
    config_data=config.VALID.DATA,
    device=config.DEVICE)
data_loader_dict['valid'] = DataLoader(
    dataset=valid_data_loader,
    num_workers=2,
    batch_size=2,
    shuffle=True,
)
print(); print()

test_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
test_data_loader = test_loader(
    name="test",
    data_path=config.TEST.DATA.DATA_PATH,
    config_data=config.TEST.DATA,
    device=config.DEVICE)
data_loader_dict['test'] = DataLoader(
    dataset=test_data_loader,
    num_workers=4,
    batch_size=3,
    shuffle=False,
)

In [ ]:
test_data_loader.input_preproc, test_data_loader.label_preproc, test_data_loader.input_keys, test_data_loader.label_keys

In [ ]:
valid_data_loader.input_preproc, valid_data_loader.label_preproc, valid_data_loader.input_keys, valid_data_loader.label_keys

In [ ]:
torch.cuda.empty_cache()

In [ ]:

def load_one_path(all_epochs_load_path, epoch_num, channels):
    
    print("RadarNet channels: ", channels)
    model = RadarNet(channels=channels).to(config.DEVICE).eval()
    load_path = os.path.join(all_epochs_load_path, f'CogPhys_rPPG_ch3_PhysNet_Epoch{epoch_num}.pth') #change second part based on model used
    
    model.load_state_dict(torch.load(load_path, map_location=config.DEVICE))
    return model



In [ ]:
def process_hr_ibi(
    all_pred,
    all_gt,
    all_participant_task_chunk_list,
    fs,
    ll_cutoff=8,
    ul_cutoff=30
):
    all_pred_hr = []
    all_gt_hr = []
    all_goodness = []

    for pred, gt, (participant_task, chunk_id_list) in zip(all_pred, all_gt, all_participant_task_chunk_list):
        #print(participant_task, chunk_id_list)

        # Normalize
        pred = (pred - np.mean(pred)) / np.std(pred)
        gt = (gt - np.mean(gt)) / np.std(gt)

        # Detrend and Goodness
        goodness = calcMetrics(pred, fs, 0.1, normalize=True)[1][0]

        # 1-D gauss blur
        # pred = custom_detrend(pred, 100)
        # gt = custom_detrend(gt, 100)
        pred = np.convolve(pred, np.ones((15)) / 15, mode='same')
        gt = np.convolve(gt, np.ones((15)) / 15, mode='same')

        # Re-normalize
        pred = (pred - np.mean(pred)) / np.std(pred)
        gt = (gt - np.mean(gt)) / np.std(gt)

        # Heart rate via PSD
        pred_hr = pulse_rate_from_power_spectral_density(pred, fs, ll_cutoff, ul_cutoff, BUTTER_ORDER=2, 
                                                        DETREND=True, WELCH=False)
        gt_hr = pulse_rate_from_power_spectral_density(gt, fs, ll_cutoff, ul_cutoff, BUTTER_ORDER=2, 
                                                        DETREND=True, WELCH=False)
    
        all_pred_hr.append(pred_hr)
        all_gt_hr.append(gt_hr)
        all_goodness.append(goodness)

        #print(f"| {pred_hr:.2f} - {gt_hr:.2f} | = {abs(pred_hr - gt_hr):.2f} bpm\t\t\t{goodness:.2f}")
        #print("-"*100)
        
    return (
        np.array(all_pred_hr),
        np.array(all_gt_hr),
        np.array(all_goodness)
    )
    
    
def get_error_metric(pred_values, gt_values):
    """
    Calculate the error metric between predicted and ground truth values.
    """
    # Calculate the mean absolute error
    mae = np.mean(np.abs(pred_values - gt_values))
    # Calculate the root mean squared error
    rmse = np.sqrt(np.mean(np.square(np.abs(pred_values - gt_values))))
    # Calculate the mean absolute percentage error
    mape = np.mean(np.abs((pred_values - gt_values) / gt_values)) * 100
    # Calculate pearson correlation coefficient
    r = np.corrcoef(pred_values, gt_values)[0, 1]
    return mae, rmse, mape, r


def evaluate_and_log_metrics(all_pred_hr, all_gt_hr, all_goodness, epoch_num, plot=False):

    all_errors = np.abs(all_pred_hr - all_gt_hr)

    # Error metrics
    mae, rmse, mape, r = get_error_metric(all_pred_hr, all_gt_hr)
    
    # Print LaTeX-style row
    print("MAE, RMSE, MAPE, r, IBI")
    print(np.round(mae,2), "&", np.round(rmse,2), "&", np.round(mape,2), "&", np.round(r,2), r"\\")

    # Plot results
    if plot:
        plt.figure(figsize=(20, 8))
        plt.subplot(1, 2, 1)
        plt.plot(all_pred_hr, all_gt_hr, 'o', alpha=0.5)
        plt.xlabel('Predicted HR')
        plt.ylabel('Ground Truth HR')
        plt.title(f'HR Prediction (Epoch {epoch_num})') 

        plt.subplot(1, 2, 2)
        plt.plot(all_goodness, all_errors, 'o', alpha=0.5)
        plt.xlabel('Goodness')
        plt.ylabel('Absolute HR Error')
        plt.title(f'Goodness vs HR Error (Epoch {epoch_num})')
        plt.show()

        plt.figure(figsize=(15, 5))
        plt.hist(np.array(all_pred_hr), bins=100, alpha=0.5, label='Predicted HR')
        plt.hist(np.array(all_gt_hr), bins=100, alpha=0.5, label='GT HR')
        plt.legend()
        plt.title(f'HR Distribution (Epoch {epoch_num})')
        plt.show()

    return np.array([mae, rmse, mape, r])



In [ ]:
metrics_table = []

all_epochs_load_path = '/home/ab227/CogPhys/runs/exp/again_resp_radar_1e-3/PreTrainedModels'
for i in range(0, len(os.listdir(all_epochs_load_path))):
    epoch_num = 5 * i - 1
    if epoch_num < 0:
        epoch_num = 0
    if epoch_num >= len(os.listdir(all_epochs_load_path)):
        break
    
    channels = config.MODEL.RADARNET.CHANNELS
    model = load_one_path(all_epochs_load_path, epoch_num, channels)

    all_pred = []
    all_gt = []
    all_participant_task_chunk_list = []
    for i in range(0, len(test_data_loader), 6):
        for j in [[0, 1], [2, 3], [4, 5]]:
            radar_matrix = []
            label = []
            participant_task_list = []
            chunk_id_list = []
            with torch.no_grad():
                for k in j:
                    radar_sample, label_sample, participant_task, chunk_id = test_data_loader[i+k]
                    participant_task_list.append(participant_task)
                    chunk_id_list.append(int(chunk_id))
                    radar_matrix.append(radar_sample.to(config.DEVICE))
                    label.extend(label_sample.squeeze(0).cpu().numpy().tolist())
                radar_matrix = torch.cat(radar_matrix, dim=0).unsqueeze(0)[...,:channels//2].permute(0, 2, 3, 1)
                radar_matrix = radar_matrix.reshape(radar_matrix.shape[0], -1, radar_matrix.shape[3])
                pred = model(radar_matrix)[0].squeeze(0).cpu().numpy()
            ##################
            assert participant_task_list[0] == participant_task_list[1]
            assert chunk_id_list[0] == chunk_id_list[1]-1
            ##################
            pred = np.array(pred)
            label = np.array(label)
            all_participant_task_chunk_list.append((participant_task_list[0], chunk_id_list)) 
            all_pred.append(pred)
            all_gt.append(label)
            print(f'epoch{epoch_num}', participant_task_list[0], chunk_id_list)
            # print("-"*100)
    
    #now calc metrics for this model epoch path
    all_pred_hr, all_gt_hr, all_goodness = process_hr_ibi(all_pred, all_gt, all_participant_task_chunk_list, fs = config.TRAIN.DATA.FS)
    metrics = evaluate_and_log_metrics(all_pred_hr, all_gt_hr, all_goodness, epoch_num)
    print(metrics)
    metrics_table.append((epoch_num, metrics))
    
print(metrics_table)

In [1]:
import pandas as pd
import numpy as np
import os

#np.save('val_epoch_compare/resp_rfnet_test.npy', np.stack([row[1] for row in metrics_table]))

loaded_metrics = np.load('val_epoch_compare/resp_rfnet_test.npy')  # shape (N, num_metrics)
all_epochs_load_path = '/home/ab227/CogPhys/runs/exp/again_resp_radar_1e-3/PreTrainedModels'
epoch_nums = []
for i in range(0, len(os.listdir(all_epochs_load_path))):
    epoch_num = 5 * i - 1
    if epoch_num < 0:
        epoch_num = 0
    if epoch_num >= len(os.listdir(all_epochs_load_path)):
        break
    epoch_nums.append(epoch_num)
metrics_table = [[epoch, row] for epoch, row in zip(epoch_nums, loaded_metrics)]



# Convert to DataFrame
df = pd.DataFrame(metrics_table, columns=["Epoch", "Metrics"])
df[['MAE', 'RMSE', 'MAPE', 'r']] = pd.DataFrame(df["Metrics"].tolist(), index=df.index)
df = df.drop(columns="Metrics")

# Optional: format float display
pd.options.display.float_format = '{:.4f}'.format

# Show the formatted table
print(df.to_string(index=False))


 Epoch    MAE   RMSE    MAPE      r
     0 2.2942 3.2628 13.1076 0.2329
     4 2.2661 3.1787 13.2072 0.2267
     9 2.2760 3.1396 13.8069 0.1806
    14 2.1462 2.9335 12.6437 0.2704
    19 2.4041 3.2740 13.7513 0.2234
    24 2.2561 3.0936 12.9510 0.2477
    29 2.1766 2.9850 12.8279 0.2583
    34 2.2509 3.0971 12.7712 0.2661
    39 2.1836 3.0062 13.0566 0.2810
    44 2.5620 3.5546 14.4854 0.1119
    49 2.2754 3.2025 12.9883 0.2246


In [ ]:
# import os
# print(len(all_pred), len(all_gt), len(all_participant_task_chunk_list))
# save_folder = "waveforms/resp/test_radar_all/"
# os.makedirs(save_folder, exist_ok=True)
# with open(os.path.join(save_folder, "pred.pickle"), 'wb') as f:
#     pickle.dump({'pred': all_pred, 'gt': all_gt, 'participant_task_chunk_id_list': all_participant_task_chunk_list}, f)
# with open(os.path.join(save_folder, "pred.pickle"), 'rb') as f:
#     data = pickle.load(f)
#     pred = data['pred']
#     gt = data['gt']
#     participant_task_chunk_id_list = data['participant_task_chunk_id_list']
# print(len(pred), len(gt), len(participant_task_chunk_id_list))